# Dashboard Data Preparation

## Objective

This notebook prepares the final dashboard-ready datasets for the Streamlit application.

The goal is to convert the project’s processed analytics outputs into clean, lightweight tables that can be loaded directly by the app without additional transformation.

The dashboard-ready files will support:

- player impact leaderboards
- lineup performance tables
- player pair synergy views
- model comparison summaries
- feature importance displays

In [24]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 200)

## Load processed outputs

We load the final processed datasets generated by the metric, modeling, and visualization notebooks.

In [25]:
player_impact = pd.read_csv("../data/processed/reliable_player_impact.csv")
lineup_perf = pd.read_csv("../data/processed/reliable_lineups.csv")
pair_perf = pd.read_csv("../data/processed/reliable_pair_performance.csv")

model_results = pd.read_csv("../data/processed/model_results.csv")
feature_importance = pd.read_csv("../data/processed/model_feature_importance.csv")

top_players = pd.read_csv("../data/processed/dashboard_top_players.csv")
top_lineups = pd.read_csv("../data/processed/dashboard_top_lineups.csv")
top_pairs = pd.read_csv("../data/processed/dashboard_top_pairs.csv")

player_impact.shape, lineup_perf.shape, pair_perf.shape

((363, 6), (3267, 8), (1618, 6))

## Create dashboard summary metrics

We create a compact KPI table that can be displayed at the top of the dashboard.

In [26]:
dashboard_summary = pd.DataFrame({
    "metric": [
        "Reliable players",
        "Reliable lineups",
        "Reliable player pairs",
        "Models evaluated",
        "Top player records",
        "Top lineup records",
        "Top pair records"
    ],
    "value": [
        len(player_impact),
        len(lineup_perf),
        len(pair_perf),
        len(model_results),
        len(top_players),
        len(top_lineups),
        len(top_pairs)
    ]
})

dashboard_summary

,metric,value
0,Reliable players,363
1,Reliable lineups,3267
2,Reliable player pairs,1618
3,Models evaluated,3
4,Top player records,15
5,Top lineup records,10
6,Top pair records,15


##  Prepare player impact table

The player table is cleaned and sorted for dashboard display.

In [27]:
stints_raw = pd.read_pickle("../data/interim/stints_multigame_validated.pkl")

stints_raw.shape

(6293, 22)

In [36]:
import sqlite3

conn = sqlite3.connect("../data/raw/nba.sqlite")

# Lookup source 1: player table
player_table_lookup = pd.read_sql(
    """
    SELECT *
    FROM player
    """,
    conn
)

# Lookup source 2: play-by-play player name columns
pbp_lookup = pd.read_sql(
    """
    SELECT DISTINCT
        player1_id, player1_name,
        player2_id, player2_name,
        player3_id, player3_name
    FROM play_by_play
    """,
    conn
)

conn.close()

lookup_records = []

# From player table
for _, row in player_table_lookup.iterrows():
    if "id" in player_table_lookup.columns and "full_name" in player_table_lookup.columns:
        lookup_records.append({
            "player_id": str(row["id"]),
            "player_name": row["full_name"]
        })

# From play-by-play player1
for _, row in pbp_lookup.iterrows():
    if pd.notna(row["player1_id"]) and pd.notna(row["player1_name"]):
        lookup_records.append({
            "player_id": str(int(row["player1_id"])),
            "player_name": row["player1_name"]
        })

    if pd.notna(row["player2_id"]) and pd.notna(row["player2_name"]):
        lookup_records.append({
            "player_id": str(int(row["player2_id"])),
            "player_name": row["player2_name"]
        })

    if pd.notna(row["player3_id"]) and pd.notna(row["player3_name"]):
        lookup_records.append({
            "player_id": str(int(row["player3_id"])),
            "player_name": row["player3_name"]
        })

player_lookup = (
    pd.DataFrame(lookup_records)
    .dropna()
    .drop_duplicates()
)

player_name_lookup = dict(
    zip(player_lookup["player_id"], player_lookup["player_name"])
)

player_lookup.head()



,player_id,player_name
0,76001,Alaa Abdelnaby
1,76002,Zaid Abdul-Aziz
2,76003,Kareem Abdul-Jabbar
3,51,Mahmoud Abdul-Rauf
4,1505,Tariq Abdul-Wahad


In [37]:
dashboard_players = player_impact.copy()

dashboard_players["player_id"] = dashboard_players["player_id"].astype(str)

dashboard_players["player_name"] = (
    dashboard_players["player_id"]
    .map(player_name_lookup)
    .fillna(dashboard_players["player_id"])
)

dashboard_players["impact_per_60_sec"] = dashboard_players["impact_per_60_sec"].round(3)

dashboard_players = dashboard_players.sort_values(
    ["impact_per_60_sec", "total_duration"],
    ascending=[False, False]
)

dashboard_players.head()

,player_id,total_net_points,total_duration,total_stints,impact_per_60_sec,stints_per_game,player_name
281,693,34,434,36,0.078,NaN,Joe Smith
265,56,101,1589,102,0.064,NaN,Gary Payton
162,2045,50,849,70,0.059,NaN,Hedo Turkoglu
168,2054,17,294,22,0.058,NaN,Jake Tsakalidis
209,280,26,454,31,0.057,NaN,Felton Spencer


## Prepare lineup performance table

The lineup table is sorted by performance and formatted for app display.

In [38]:
dashboard_lineups = lineup_perf.copy()

dashboard_lineups = dashboard_lineups.sort_values(
    ["net_per_60", "total_duration"],
    ascending=[False, False]
)

dashboard_lineups["net_per_60"] = dashboard_lineups["net_per_60"].round(2)

dashboard_lineups.head()

,home_lineup_tuple,away_lineup_tuple,total_home_points,total_away_points,total_duration,total_stints,net_points,net_per_60
2664,"('1717', '363', '458', '919', '93')","('123', '1505', '1721', '1899', '686')",6,0,7,1,6,51.43
1333,"('1499', '1559', '179', '2040', '677')","('146', '1630', '182', '2043', '376')",5,0,6,1,5,50.00
2971,"('1890', '201', '2091', '2092', '2101')","('2042', '2059', '223', '458', '762')",4,0,5,1,4,48.00
2988,"('1890', '201', '2092', '2101', '915')","('1749', '1883', '1903', '2044', '983')",5,0,7,1,5,42.86
2659,"('1717', '363', '458', '714', '762')","('209', '213', '349', '722', '788')",8,0,12,1,8,40.00


##  Prepare player pair synergy table

The pair synergy table is sorted by impact and total duration to prioritize effective and consistently used pairings.

In [39]:
dashboard_pairs = pair_perf.copy()

dashboard_pairs["player_1"] = dashboard_pairs["player_1"].astype(str)
dashboard_pairs["player_2"] = dashboard_pairs["player_2"].astype(str)

dashboard_pairs["player_1_name"] = dashboard_pairs["player_1"].map(player_name_lookup)
dashboard_pairs["player_2_name"] = dashboard_pairs["player_2"].map(player_name_lookup)

dashboard_pairs["player_1_name"] = dashboard_pairs["player_1_name"].fillna(dashboard_pairs["player_1"])
dashboard_pairs["player_2_name"] = dashboard_pairs["player_2_name"].fillna(dashboard_pairs["player_2"])

dashboard_pairs["pair_name"] = (
    dashboard_pairs["player_1_name"] + " + " + dashboard_pairs["player_2_name"]
)

dashboard_pairs["impact_per_60"] = dashboard_pairs["impact_per_60"].round(2)

dashboard_pairs = dashboard_pairs.sort_values(
    ["impact_per_60", "total_duration"],
    ascending=[False, False]
)

dashboard_pairs.head()

,player_1,player_2,total_net_points,total_duration,total_stints,impact_per_60,player_1_name,player_2_name,pair_name
1363,280,703,30,205,12,0.15,Felton Spencer,Kurt Thomas,Felton Spencer + Kurt Thomas
949,1887,417,35,285,27,0.12,Wally Szczerbiak,Sam Mitchell,Wally Szczerbiak + Sam Mitchell
28,1023,56,33,269,18,0.12,Emanual Davis,Gary Payton,Emanual Davis + Gary Payton
168,121,56,105,927,54,0.11,Patrick Ewing,Gary Payton,Patrick Ewing + Gary Payton
922,1884,1924,35,322,20,0.11,Baron Davis,Lee Nailon,Baron Davis + Lee Nailon


## Prepare modeling summary tables

Model comparison and feature importance tables are formatted for dashboard display.

In [40]:
dashboard_model_results = model_results.copy()

for col in ["rmse", "mae", "r2"]:
    if col in dashboard_model_results.columns:
        dashboard_model_results[col] = dashboard_model_results[col].round(3)

dashboard_model_results

,model,rmse,mae,r2
0,Linear Regression,6.900,5.466,0.042
1,Ridge Regression,6.981,5.517,0.020
2,Random Forest,6.950,5.476,0.028


In [41]:
dashboard_feature_importance = feature_importance.copy()

if "importance" in dashboard_feature_importance.columns:
    dashboard_feature_importance["importance"] = dashboard_feature_importance["importance"].round(4)

if "coefficient" in dashboard_feature_importance.columns:
    dashboard_feature_importance["coefficient"] = dashboard_feature_importance["coefficient"].round(4)

dashboard_feature_importance.head(15)

,feature,coefficient
0,away_std_player_impact,247.2696
1,away_min_player_impact,71.4468
2,star_gap,61.9080
3,max_impact_diff,61.9080
4,std_player_impact,33.1889
5,min_player_impact,25.1916
6,max_player_impact,11.8945
7,away_avg_player_impact,1.9675
8,total_duration,-0.0041
9,total_stints,-0.0090


## Preview final dashboard datasets

Before saving, we preview the dashboard-ready tables to confirm that the outputs are clean, readable, and app-ready.

In [42]:
display(dashboard_summary)
display(dashboard_players.head())
display(dashboard_lineups.head())
display(dashboard_pairs.head())
display(dashboard_model_results)
display(dashboard_feature_importance.head())

,metric,value
0,Reliable players,363
1,Reliable lineups,3267
2,Reliable player pairs,1618
3,Models evaluated,3
4,Top player records,15
5,Top lineup records,10
6,Top pair records,15


,player_id,total_net_points,total_duration,total_stints,impact_per_60_sec,stints_per_game,player_name
281,693,34,434,36,0.078,NaN,Joe Smith
265,56,101,1589,102,0.064,NaN,Gary Payton
162,2045,50,849,70,0.059,NaN,Hedo Turkoglu
168,2054,17,294,22,0.058,NaN,Jake Tsakalidis
209,280,26,454,31,0.057,NaN,Felton Spencer


,home_lineup_tuple,away_lineup_tuple,total_home_points,total_away_points,total_duration,total_stints,net_points,net_per_60
2664,"('1717', '363', '458', '919', '93')","('123', '1505', '1721', '1899', '686')",6,0,7,1,6,51.43
1333,"('1499', '1559', '179', '2040', '677')","('146', '1630', '182', '2043', '376')",5,0,6,1,5,50.00
2971,"('1890', '201', '2091', '2092', '2101')","('2042', '2059', '223', '458', '762')",4,0,5,1,4,48.00
2988,"('1890', '201', '2092', '2101', '915')","('1749', '1883', '1903', '2044', '983')",5,0,7,1,5,42.86
2659,"('1717', '363', '458', '714', '762')","('209', '213', '349', '722', '788')",8,0,12,1,8,40.00


,player_1,player_2,total_net_points,total_duration,total_stints,impact_per_60,player_1_name,player_2_name,pair_name
1363,280,703,30,205,12,0.15,Felton Spencer,Kurt Thomas,Felton Spencer + Kurt Thomas
949,1887,417,35,285,27,0.12,Wally Szczerbiak,Sam Mitchell,Wally Szczerbiak + Sam Mitchell
28,1023,56,33,269,18,0.12,Emanual Davis,Gary Payton,Emanual Davis + Gary Payton
168,121,56,105,927,54,0.11,Patrick Ewing,Gary Payton,Patrick Ewing + Gary Payton
922,1884,1924,35,322,20,0.11,Baron Davis,Lee Nailon,Baron Davis + Lee Nailon


,model,rmse,mae,r2
0,Linear Regression,6.900,5.466,0.042
1,Ridge Regression,6.981,5.517,0.020
2,Random Forest,6.950,5.476,0.028


,feature,coefficient
0,away_std_player_impact,247.2696
1,away_min_player_impact,71.4468
2,star_gap,61.9080
3,max_impact_diff,61.9080
4,std_player_impact,33.1889


## Save dashboard-ready datasets

The final dashboard files are saved in the processed data folder. These files are designed to be loaded directly by the Streamlit app.

In [43]:
dashboard_summary.to_csv("../data/processed/dashboard_summary.csv", index=False)
dashboard_players.to_csv("../data/processed/dashboard_players.csv", index=False)
dashboard_lineups.to_csv("../data/processed/dashboard_lineups.csv", index=False)
dashboard_pairs.to_csv("../data/processed/dashboard_pairs.csv", index=False)
dashboard_model_results.to_csv("../data/processed/dashboard_model_results.csv", index=False)
dashboard_feature_importance.to_csv("../data/processed/dashboard_feature_importance.csv", index=False)

print("Dashboard-ready datasets saved successfully.")

Dashboard-ready datasets saved successfully.


## Saved dashboard files

The following files were created for the Streamlit dashboard.

In [44]:
saved_files = [
    "dashboard_summary.csv",
    "dashboard_players.csv",
    "dashboard_lineups.csv",
    "dashboard_pairs.csv",
    "dashboard_model_results.csv",
    "dashboard_feature_importance.csv"
]

print("Dashboard-ready datasets saved successfully.")
saved_files

Dashboard-ready datasets saved successfully.


['dashboard_summary.csv',
 'dashboard_players.csv',
 'dashboard_lineups.csv',
 'dashboard_pairs.csv',
 'dashboard_model_results.csv',
 'dashboard_feature_importance.csv']

In [45]:
dashboard_pairs[["player_1", "player_1_name", "player_2", "player_2_name", "pair_name"]].head(20)

,player_1,player_1_name,player_2,player_2_name,pair_name
1363,280,Felton Spencer,703,Kurt Thomas,Felton Spencer + Kurt Thomas
949,1887,Wally Szczerbiak,417,Sam Mitchell,Wally Szczerbiak + Sam Mitchell
28,1023,Emanual Davis,56,Gary Payton,Emanual Davis + Gary Payton
168,121,Patrick Ewing,56,Gary Payton,Patrick Ewing + Gary Payton
922,1884,Baron Davis,1924,Lee Nailon,Baron Davis + Lee Nailon
1345,275,Allan Houston,280,Felton Spencer,Allan Houston + Felton Spencer
1413,349,Mark Jackson,895,Tyrone Corbin,Mark Jackson + Tyrone Corbin
1574,722,Corliss Williamson,891,Charles Oakley,Corliss Williamson + Charles Oakley
771,1732,Felipe Lopez,685,Cherokee Parks,Felipe Lopez + Cherokee Parks
592,168,Chris Mills,682,Bob Sura,Chris Mills + Bob Sura


## Key findings

This notebook packages the project’s final analytics outputs into dashboard-ready datasets.

The resulting files provide clean and lightweight inputs for:

- player impact exploration
- lineup performance analysis
- player pair synergy evaluation
- model comparison
- feature importance interpretation

By separating dashboard data preparation from app logic, the Streamlit application can remain faster, cleaner, and easier to maintain.

## Next step

The next stage is to update the Streamlit dashboard so it consumes these multi-game dashboard-ready files and displays the final analytics outputs interactively.